In [ ]:
import os
os.chdir("..")

In [ ]:
from prfood_repl import FoodDeseart
import hashlib
import tempfile
from jp_tools import download
import geopandas as gpd
from pathlib import Path

fd = FoodDeseart()

In [ ]:
file_path = Path("data/") / "external" / "geo-zipcode.parquet"

# Create a unique hash for the temp file based on the target path
name_hash = hashlib.md5(str(file_path).encode()).hexdigest()
temp_zip = Path(tempfile.gettempdir()) / f"{name_hash}.zip"
temp_zip

In [ ]:
df = fd.zips_goem()
df

In [ ]:
# download(
#                 url="https://www2.census.gov/geo/tiger/TIGER2024/ZCTA520/tl_2024_us_zcta520.zip",
#                 filename="delete.zip",
#             )
# Process files
gdf = gpd.read_file("delete.zip")
gdf = gdf.rename(columns={"ZCTA5CE20": "zipcode"})
gdf = gdf[gdf["zipcode"].str.startswith("00")].reset_index(drop=True)
gdf = gdf[["zipcode", "geometry"]]
gdf["zipcode"] = gdf["zipcode"].str.strip()
gdf = gdf[~gdf["zipcode"].isin(["00820", "00850", "00851"])]
gdf = gdf.to_crs(epsg=5070)
gdf = gdf.explode(ignore_index=True)
connected_mask = gdf.geometry.apply(
    lambda geom: gdf.geometry.intersects(geom).sum() > 1
)
gdf = gdf.loc[connected_mask]
gdf = gdf.dissolve(by="zipcode", as_index=False)
gdf.plot()

